# Глава 4. Градиентный спуск

Код к главе 4 книги 1: спуск по одному параметру при разных скоростях обучения,
векторизованный спуск для линейной регрессии (листинг 4.1), влияние масштаба признаков,
стохастический и мини-пакетный спуск, `SGDRegressor` (листинг 4.2).

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markpolyak/ml-textbook-code/blob/main/book1/ch04_gd.ipynb)

In [ ]:
# В Colab ставим пакет mlbook с данными книги (локально: pip install -e . в корне репозитория)
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/markpolyak/ml-textbook-code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlbook import flats

## Один параметр

Точки $(1; 1)$, $(2; 2)$, $(3; 3)$, модель $g(x) = \theta_1 x$:
$Q(\theta_1) = \frac{14}{3}(\theta_1 - 1)^2$, $Q'(\theta_1) = \frac{28}{3}(\theta_1 - 1)$.
Шаг: $\theta_1 := \theta_1 - \eta\, Q'(\theta_1)$.

In [ ]:
Q = lambda t: 14 / 3 * (t - 1) ** 2
dQ = lambda t: 28 / 3 * (t - 1)


def steps(eta, t0=0.0, n=4):
    t = [t0]
    for _ in range(n):
        t.append(t[-1] - eta * dQ(t[-1]))
    return np.array(t)


for eta in (0.05, 0.2, 0.25):
    print(f"η = {eta}: множитель {1 - 28 * eta / 3:.2f}, θ1 =", steps(eta).round(3))

При $\eta = 0{,}05$ спуск сходится плавно, при $0{,}2$ — колеблясь, при $0{,}25$ расходится:
граница сходимости $\eta < 2/Q'' = 2 \cdot 3/28 \approx 0{,}21$ (рис. 4.1).

In [ ]:
ts = np.linspace(-1.5, 3.5, 200)
fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, eta in zip(axes, (0.05, 0.2, 0.25)):
    t = steps(eta)
    ax.plot(ts, Q(ts), color="0.6")
    ax.plot(t, Q(t), "o-", color="C1")
    ax.set_title(f"η = {eta}")
    ax.set_xlabel(r"$\theta_1$")
    ax.set_ylim(0, 22)
axes[0].set_ylabel(r"$Q(\theta_1)$")
plt.show()

## Градиентный спуск для линейной регрессии

Листинг 4.1: шаг $\boldsymbol{\theta} := \boldsymbol{\theta} - \frac{2\eta}{\ell}
\mathbf{F}^{\mathrm{T}}(\mathbf{F}\boldsymbol{\theta} - \mathbf{y})$ одной строкой NumPy.

In [ ]:
import numpy as np

def gradient_descent(F, y, eta=0.1, steps=1000, tol=1e-6):
    ell, p = F.shape                  # F — со столбцом f0 = 1
    theta = np.zeros(p)
    for t in range(steps):
        grad = 2 / ell * F.T @ (F @ theta - y)
        theta = theta - eta * grad    # все параметры сразу
        if np.linalg.norm(grad) < tol:
            break
    return theta

Точки $(1; 1)$, $(2; 3)$, $(3; 2)$ из гл. 3: спуск приходит в решение МНК $(1;\ 0{,}5)$.

In [ ]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([1.0, 3.0, 2.0])
F = np.column_stack([np.ones(3), x])
print(gradient_descent(F, y, eta=0.1, steps=10_000).round(4))

## Масштаб признаков

25 шагов с $\eta = 0{,}15$ по исходному признаку и по стандартизованному (рис. 4.2).
Линии уровня по исходному признаку — вытянутые эллипсы, $\kappa(\mathbf{F}^{\mathrm{T}}
\mathbf{F}) \approx 46$; после стандартизации столбцы ортогональны и $\kappa = 1$.

In [ ]:
def trajectory(F, start, eta=0.15, n=25):
    th = [np.array(start, float)]
    for _ in range(n):
        th.append(th[-1] - eta * 2 / len(y) * F.T @ (F @ th[-1] - y))
    return np.array(th)


z = (x - x.mean()) / x.std()
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, feat, start, title in [(axes[0], x, (-0.5, 1.5), "а) признак x"),
                               (axes[1], z, (0.5, 1.5), "б) стандартизованный признак")]:
    Fs = np.column_stack([np.ones(3), feat])
    th = trajectory(Fs, start)
    opt = np.linalg.lstsq(Fs, y, rcond=None)[0]
    d0, d = np.linalg.norm(th[0] - opt), np.linalg.norm(th[-1] - opt)
    print(f"{title}: κ = {np.linalg.cond(Fs.T @ Fs):.0f}, расстояние до минимума",
          f"{d0:.2f} → {d:.1e} ({d / d0:.0%})")
    T0, T1 = np.meshgrid(np.linspace(opt[0] - 2, opt[0] + 2, 300),
                         np.linspace(opt[1] - 1.2, opt[1] + 1.2, 300))
    Qg = ((T0[..., None] + T1[..., None] * feat - y) ** 2).mean(axis=-1)
    ax.contour(T0, T1, Qg, levels=[0.55, 0.8, 1.3, 2, 3, 4.5], colors="C0", linewidths=0.7)
    ax.plot(th[:, 0], th[:, 1], "o-", color="C1", markersize=3)
    ax.plot(*opt, "*", color="black", markersize=10)
    ax.set_title(title)
    ax.set_xlabel(r"$\theta_0$")
    ax.set_aspect("equal")
axes[0].set_ylabel(r"$\theta_1$")
plt.show()

## Полный, мини-пакетный и стохастический спуск

Синтетическая задача: $\ell = 1000$ объектов, пять признаков. За эпоху полный градиент
делает один шаг, мини-пакетный — 50, стохастический — 1000 (рис. 4.3).

In [ ]:
rng = np.random.default_rng(0)
L, N = 1000, 5
Fb = np.column_stack([np.ones(L), rng.normal(size=(L, N))])
yb = Fb @ rng.normal(size=N + 1) + rng.normal(0, 0.5, L)
Qb = lambda th: np.mean((Fb @ th - yb) ** 2)
Qmin = Qb(np.linalg.lstsq(Fb, yb, rcond=None)[0])


def run(batch, eta, epochs=5):
    th = np.zeros(N + 1)
    xs, qs, seen = [0.0], [Qb(th)], 0
    for _ in range(epochs):
        idx = rng.permutation(L)              # перемешиваем и идём по порядку
        for s in range(0, L, batch):
            b = idx[s:s + batch]
            th = th - eta * 2 / len(b) * Fb[b].T @ (Fb[b] @ th - yb[b])
            seen += len(b)
            if batch == L or seen % 20 == 0:
                xs.append(seen / L)
                qs.append(Qb(th))
    return np.array(xs), np.array(qs)


fig, ax = plt.subplots(figsize=(6, 4))
for batch, eta, lab in [(L, 0.3, "полный градиент, η = 0,3"),
                        (20, 0.05, "мини-пакеты по 20, η = 0,05"),
                        (1, 0.01, "стохастический, η = 0,01")]:
    xs, qs = run(batch, eta)
    ax.plot(xs, qs, label=lab, marker="o" if batch == L else None)
    print(f"{lab}: Q после первой эпохи {qs[np.searchsorted(xs, 1.0)]:.3f}, минимум {Qmin:.3f}")
ax.axhline(Qmin, color="0.5", linewidth=0.8)
ax.set_yscale("log")
ax.set_xlabel("эпохи")
ax.set_ylabel(r"$Q(\boldsymbol{\theta})$")
ax.legend()
plt.show()

**Оценка $Q$ на лету** экспоненциальным скользящим средним: $\bar Q := \beta\,\mathcal{L}_i +
(1 - \beta)\,\bar Q$. При $\beta = 0{,}01$ оценка — примерно средняя потеря на последней
сотне объектов. Сравним её с настоящим $Q$ по всей выборке.

In [ ]:
th, Qbar, beta = np.zeros(N + 1), None, 0.01
for i in rng.permutation(L):
    r = Fb[i] @ th - yb[i]
    Qbar = r ** 2 if Qbar is None else beta * r ** 2 + (1 - beta) * Qbar   # потеря до шага
    th = th - 0.01 * 2 * r * Fb[i]
print(f"оценка на лету {Qbar:.3f}, Q по всей выборке {Qb(th):.3f}")

## SGDRegressor

Листинг 4.2 на данных о квартирах: признаки — площадь, этаж и расстояние до метро.

In [ ]:
from sklearn.model_selection import train_test_split

df = flats.load()
X = df[["площадь", "этаж", "метро"]]
y = df["цена"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    SGDRegressor(penalty=None, eta0=0.01, max_iter=1000,
                 tol=1e-4, random_state=0))
model.fit(X_train, y_train)

Сравним с точным решением МНК в том же конвейере: веса почти совпадают.

In [ ]:
from sklearn.linear_model import LinearRegression

exact = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
print("SGD:", model[-1].coef_.round(3), round(model[-1].intercept_[0], 3),
      " эпох:", model[-1].n_iter_)
print("МНК:", exact[-1].coef_.round(3), round(exact[-1].intercept_, 3))

## Задачи

**1.** Два шага из $\theta_1 = 2$ при $\eta = 0{,}1$ для модели $g(x) = \theta_1 x$.

In [ ]:
print(steps(0.1, t0=2.0, n=2).round(3))

**3.** Один шаг полного спуска из $(0;\ 0)$ при $\eta = 0{,}1$ для точек $(1; 1)$, $(2; 3)$,
$(3; 2)$ и модели $\theta_0 + \theta_1 x$.

In [ ]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([1.0, 3.0, 2.0])
F = np.column_stack([np.ones(3), x])
grad = 2 / 3 * F.T @ (F @ np.zeros(2) - y)
print("∇Q =", grad.round(2), " θ =", (-0.1 * grad).round(2))

**4.** Шаг стохастического спуска из минимума $Q$, $\boldsymbol{\theta} = (1;\ 0{,}5)$,
по объекту $(2; 3)$ при $\eta = 0{,}1$: градиент потери на одном объекте в минимуме $Q$
не равен нулю.

In [ ]:
theta = np.array([1.0, 0.5])
xi, yi = np.array([1.0, 2.0]), 3.0
g = 2 * (theta @ xi - yi) * xi
print("градиент:", g, " θ =", theta - 0.1 * g)